# Agentic Code Reviewer -- notebook demo

This notebook is a runnable demo of the **Agentic Code Reviewer** project from
the course: [`docs/projects/agentic-code-reviewer/`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/agentic-code-reviewer),
companion to the fuller local CLI at [`examples/agentic-code-reviewer/review.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/agentic-code-reviewer/review.py).

It captures a real `git diff`, hands it to a free-tier LLM with a code-review
system prompt, and prints back structured, actionable feedback -- file, category,
severity, explanation, and a suggested fix.

## A note on running this in a notebook

The real version of this tool (`examples/agentic-code-reviewer/review.py`) reviews
**your own local git repository** -- your uncommitted changes, or a commit from
a repo you already have cloned on disk. That's the whole point of the tool: a fast
first-pass review of work you're actually doing.

Colab, Kaggle, and Binder don't give you that -- there's no local repo of your own
to diff here, just this notebook's ephemeral, ready-made environment. So **this demo
adapts the tool**: instead of diffing your own work, it shallow-clones this course's
own repository right here in the notebook and reviews one real, small, historical
commit from it with `git show`. That's a legitimate demo of every piece of the tool
(the `subprocess` diff capture, the system prompt, the LLM call, the structured
output) even though it isn't reviewing work you personally wrote.

**Locally, or in a GitHub Codespace, you'd point this at your own repo instead** --
see the [project walkthrough](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/agentic-code-reviewer)
for that path; it's still the recommended way to review your own actual work.

In [ ]:
!pip install -q openai

## Get a real diff to review

Shallow-clone this course's own repository and pick one real, small, illustrative
commit from its actual history: a CSS bug fix (`be914ee`, "Fix hamburger menu not
opening on desktop browsers") that touches one file with a focused, explainable bug
-- a good size and shape for a demo review.

In [ ]:
!git clone --depth 50 https://github.com/abderrahim-lectures/python-data-analysis-course.git /tmp/repo

In [ ]:
import subprocess

DEMO_COMMIT = "be914ee44ce104124f38b775ff200072e91095bd"  # "Fix hamburger menu not opening on desktop browsers"


def get_diff_for_commit(commit: str, repo_dir: str = "/tmp/repo") -> str:
    """The diff introduced by one specific past commit, vs. its parent.

    Same idea as `get_diff_for_commit` in review.py, just pointed at the
    freshly-cloned repo above instead of the current directory.
    """
    result = subprocess.run(
        ["git", "-C", repo_dir, "show", commit],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(f"git show {commit} failed:\n{result.stderr}")
    return result.stdout


demo_diff = get_diff_for_commit(DEMO_COMMIT)
print(demo_diff[:1500])

## The code-review system prompt

This is the exact `SYSTEM_PROMPT` from `review.py` -- it's what turns a
general-purpose chat model into a focused, structured code reviewer: what to
look for, what to ignore, and what shape its answer should take.

In [ ]:
SYSTEM_PROMPT = """\
You are an experienced, pragmatic senior software engineer doing a code review.
You will be given a unified git diff. Review ONLY what the diff actually
changes -- do not comment on surrounding code you can't see, and do not
invent context that isn't in the diff.

For each issue you find, report:
- file and, if visible in the diff's @@ hunk header, the approximate line
- category: one of Bug, Style, Missing Test, Unclear Naming, Security, Other
- severity: Critical, Warning, or Suggestion
- a short, concrete explanation of the issue
- a specific suggested fix, not just "consider improving this"

Focus on:
- likely bugs (off-by-one errors, unhandled edge cases, wrong operators,
  mutated shared state)
- style inconsistencies with the surrounding code
- missing or clearly inadequate test coverage for the change
- unclear variable/function names that would confuse the next reader
- obvious security issues (secrets, injection, unsafe deserialization)

If the diff genuinely has no issues, say so plainly and briefly -- do not
invent problems just to have something to say. Never respond with just
"looks good" and nothing else; always state what you checked.

Format your response as a numbered list of issues (or a short "no issues
found, because ..." paragraph), not prose paragraphs.
"""

## Get a free-tier API key

This demo defaults to **GitHub Models** -- free, no separate signup, just a
personal access token with the `models: read` scope from
[github.com/settings/tokens](https://github.com/settings/tokens). Any of the
other five providers wired up in `review.py` (Gemini, Groq, Mistral, Cerebras,
OpenRouter) work too -- see that file's `PROVIDERS` dict for their base URLs
and env var names, and adjust `LLM_PROVIDER` below.

The key is entered with `getpass` so it never gets typed into a visible cell
or saved into this notebook's output -- never hardcode a real API key here.

In [ ]:
import os
from getpass import getpass

LLM_PROVIDER = "github"  # change to gemini / groq / mistral / cerebras / openrouter if you prefer
os.environ["GITHUB_TOKEN"] = getpass("Enter your free-tier GitHub Models token (GITHUB_TOKEN): ")

## The review logic itself

This mirrors `truncate_diff`, `PROVIDERS`, and `review_diff` from `review.py`
directly -- the same truncation cap, the same free-tier providers (all exposed
through the `openai` client, just pointed at each provider's own
OpenAI-compatible endpoint), and the same call shape.

In [ ]:
from openai import OpenAI

MAX_DIFF_CHARS = 12_000


def truncate_diff(diff: str, max_chars: int = MAX_DIFF_CHARS) -> str:
    """Cuts an oversized diff down to a size that fits a free-tier context window."""
    if len(diff) <= max_chars:
        return diff
    return diff[:max_chars] + f"\n\n... [diff truncated -- {len(diff) - max_chars} more characters not shown] ..."


def _build_github_client() -> OpenAI:
    return OpenAI(api_key=os.environ["GITHUB_TOKEN"], base_url="https://models.github.ai/inference")


def _build_gemini_client() -> OpenAI:
    return OpenAI(
        api_key=os.environ["GOOGLE_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )


def _build_groq_client() -> OpenAI:
    return OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")


def _build_mistral_client() -> OpenAI:
    return OpenAI(api_key=os.environ["MISTRAL_API_KEY"], base_url="https://api.mistral.ai/v1")


def _build_cerebras_client() -> OpenAI:
    return OpenAI(api_key=os.environ["CEREBRAS_API_KEY"], base_url="https://api.cerebras.ai/v1")


def _build_openrouter_client() -> OpenAI:
    return OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1")


PROVIDERS = {
    "github": (_build_github_client, "gpt-4o-mini"),
    "gemini": (_build_gemini_client, "gemini-3.5-flash"),
    "groq": (_build_groq_client, "llama-3.3-70b-versatile"),
    "mistral": (_build_mistral_client, "mistral-small-latest"),
    "cerebras": (_build_cerebras_client, "llama-3.3-70b"),
    "openrouter": (_build_openrouter_client, "meta-llama/llama-3.3-70b-instruct:free"),
}


def review_diff(diff: str, provider: str = LLM_PROVIDER) -> str:
    """Sends a diff to a free-tier LLM with the code-review system prompt and returns its feedback."""
    if not diff.strip():
        return "No changes to review -- the diff is empty."

    if provider not in PROVIDERS:
        raise ValueError(f"Unknown provider '{provider}'. Choose one of: {', '.join(PROVIDERS)}")
    build_client, model = PROVIDERS[provider]
    client = build_client()

    diff = truncate_diff(diff)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Review this diff:\n\n```diff\n{diff}\n```"},
        ],
    )
    return response.choices[0].message.content

## Run the review

Reviewing the demo commit cloned above -- a real historical commit from this
course's own repository, not the student's own local work (see the note near
the top of this notebook).

In [ ]:
print(f"Reviewing {len(demo_diff)} characters of diff from commit {DEMO_COMMIT[:7]}...\n")
feedback = review_diff(demo_diff)
print(feedback)